<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/CenterPointHead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
#drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
CUDA_version = torch.version.cuda.replace('.', '')

!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

import torch_sparse

from scipy.ndimage import maximum_filter

import sys

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 52.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 131.2 MB/s eta 0:00:00


In [3]:
!npx degit google-research-datasets/Objectron/objectron objectron

sys.path.insert(0, '/content')

from objectron.dataset.iou import IoU as IoU3d
from objectron.dataset.box import Box

In [ ]:
def generate_heatmap(w,h, centers,sigma):

  x_axis = torch.arange(0, w, device=centers.device, dtype=torch.float32)
  y_axis = torch.arange(0, h, device=centers.device, dtype=torch.float32)
  y_grid, x_grid = torch.meshgrid(y_axis, x_axis, indexing='ij')

  distances_squared = torch.zeros((w,h), device=centers.device)

  for i in range(centers.shape[0]):
    distances_squared += (x_grid-centers[i][0])**2 +(y_grid-centers[i][1])

  heatmap = torch.exp(-1*distances_squared/(2*sigma**2))

  return heatmap

In [74]:
class CenterPoint_FirstStage(nn.Module): #similar classes(num_classes) are grouped together into one task head. The classes should have similar size, z, yaw, etc. so one task head can be used
#input is assumed to be of shape [C,W,H]
  def __init__(self, w, h, C, num_classes, K, hidden_channels=64, stride = 2):
    super().__init__()

    self.w = w
    self.h = h
    self.num_classes = num_classes
    self.C = C
    self.K = K #K is max number of centers per heatmap layer
    self.stride = stride

    self.shared = nn.Sequential(nn.Conv2d(self.C, hidden_channels, kernel_size = 3, padding = 1, bias = True), nn.ReLU())

    self.heat_conv = nn.Conv2d(hidden_channels, num_classes, kernel_size = 1, stride = self.stride)

    self.offset = nn.Conv2d(hidden_channels, 2, kernel_size = 1)             # (δx, δy)
    self.z = nn.Conv2d(hidden_channels, 1, kernel_size = 1)             # z center
    self.size = nn.Conv2d(hidden_channels, 3, kernel_size = 1)             # log(w, l, h)
    self.yaw = nn.Conv2d(hidden_channels, 2, kernel_size = 1)             # (sin θ, cos θ)
    #self.velocity = nn.Conv2d(hidden_channels, 2, kernel_size = 1)


  def get_shared_feats(self, feature_map):
    return self.shared(feature_map)

  def learn_heatmap(self, feature_map):

    heatmap = self.shared(feature_map)
    heatmap = F.sigmoid(self.heat_conv(heatmap)).permute(1,2,0)

    return heatmap

  def get_centers(self, heatmap):
    centers_list = []

    for k in range(self.num_classes):
      heatmap_k = heatmap[:,:, k].cpu().detach().numpy()

      local_max_filter = maximum_filter(heatmap_k, size=3, mode='constant', cval=-np.inf)

      is_local_max = (heatmap_k == local_max_filter)

      rows, cols = np.where(is_local_max)

      for i in range(rows.size):

        layer_center_list = []
        I_list = []

        I = heatmap_k[rows[i]][cols[i]]

        center_0 = torch.tensor([rows[i],cols[i],I,k])

        I_list.append(I)
        layer_center_list.append(center_0)

      while len(layer_center_list) > self.K:
        j = I_list.index(min(I_list))
        layer_center_list.remove(layer_center_list[j])

      layer_center_list = torch.stack(layer_center_list)

      centers_list.append(layer_center_list)

    centers = torch.cat(centers_list, dim = 0)

    return centers

  def regression_heads(self, feature_map):

    shared_feats = self.shared(feature_map)

    offset = self.offset(shared_feats).permute(1,2,0)
    z = self.z(shared_feats).squeeze()
    size = self.size(shared_feats).permute(1,2,0)
    yaw = self.yaw(shared_feats).permute(1,2,0)
    #velocity = self.velocity(shared_feats).permute(1,2,0)

    return offset, z, size, yaw

  def forward(self, feature_map):

    heatmap = self.learn_heatmap(feature_map)

    offset, z, size, yaw = self.regression_heads(feature_map)

    centers = self.get_centers(heatmap)

    out_list = []

    for i in range(centers.shape[0]):

      cx = int(centers[i][0]) * self.stride
      cy = int(centers[i][1]) * self.stride
      I = centers[i][2]
      k = centers[i][3]

      offset_0 = offset[cx][cy]
      z_0 = z[cx][cy].detach()
      size_0 = size[cx][cy]
      yaw_0 = yaw[cx][cy]

      centers_0 = torch.tensor([cx+offset_0[0].item(), cy+offset_0[1].item()], dtype = offset_0.dtype, device = offset_0.device)

      temp_tensor = torch.tensor([z_0, k, I], dtype = offset_0.dtype, device = offset_0.device)

      out_0 = [size_0, yaw_0, centers_0, temp_tensor] #test first

      out_0 = torch.cat(out_0)

      out_list.append(out_0)

    out = torch.stack(out_list, dim = 0)

    return out



In [75]:
class CenterPoint_SecondStage(nn.Module):

  def __init__(self, w, h, C, num_classes, hidden_channels = 32): #C is C of shared map

    super().__init__()

    self.w = w
    self.h = h
    self.C =C
    self.num_classes = num_classes

    self.refine_network = nn.Sequential(nn.Linear(5*C, hidden_channels, bias = True),
                                        nn.ReLU(),
                                        nn.Linear(hidden_channels,8))

  def get_box_centers(self, size, yaw, cx, cy):

    center = torch.tensor([cx.item(),cy.item()], dtype = size.dtype, device= size.device)

    lwh = torch.exp(size)
    l_1 = (yaw * lwh[0]*0.5)
    l_2 = l_1 * -1
    l_3 = (yaw * lwh[1]*0.5)
    l_4 = l_3 * -1

    l_1, l_2, l_3, l_4 = l_1+center, l_2+center, l_3+center, l_4+center

    center_list = [center,l_1,l_2,l_3,l_4]

    return torch.stack(center_list)

  def BEV_sample(self, box_centers, shared_map): #box centers should be batched - [N, 5, 2]

    C = self.C
    N = box_centers.shape[0]

    max_x = torch.max(box_centers[:,:,0])
    max_y = torch.max(box_centers[:,:,1])

    min_x = torch.min(box_centers[:,:,0])
    min_y = torch.min(box_centers[:,:,1])

    normalized_x = 2*((box_centers[:,:,0]+min_x)/(max_x+min_x)) -1
    normalized_y = 2*((box_centers[:,:,1]+min_y)/(max_y+min_y)) -1

    normalized_centers = torch.stack([normalized_x, normalized_y]).permute(1,2,0)

    sampled = F.grid_sample(shared_map.unsqueeze(0), normalized_centers.unsqueeze(0), align_corners=True)

    out_feats = sampled.view(N, 5*C)

    return out_feats

  def refine_boxes(self, sampled_centers, original_boxes):

    out_list = []

    out_refine = self.refine_network(sampled_centers)

    dx, dy, dz, dlogl, dlogw, dlogh, dyaw, I = out_refine.T.detach()

    for i in range(original_boxes.shape[0]):

      d_vec = torch.tensor([dlogl[i],dlogw[i],dlogh[i],0,0,dx[i],dy[i],dz[i],0,0], dtype = original_boxes.dtype, device = original_boxes.device)

      alpha = torch.arcsin(original_boxes[i][3])
      alpha = alpha+dyaw[i]
      sin_0 = torch.sin(alpha)
      cos_0 = torch.cos(alpha)

      I_0 = torch.sqrt(I[i]*original_boxes[i][9])

      if I_0 != I_0:
        I_0 = torch.zeros((1))

      out_0 = original_boxes[i] + d_vec

      out_0[3] = sin_0.item()
      out_0[4] = cos_0.item()
      out_0[9] = I_0.item()

      out_list.append(out_0)

    out = torch.stack(out_list)

    return out

  def forward(self, centers_in, shared_map):
    box_center_list = []

    cx = centers_in[:, 5]
    cy = centers_in[:, 6]

    yaws = centers_in[:, 3:5]

    size = centers_in[:, 0:3]

    for i in range(centers_in.shape[0]):

      size_0 = size[i]

      yaws_0 = yaws[i]

      cx_0 = cx[i]
      cy_0 = cy[i]

      box_centers_0 = self.get_box_centers(size_0, yaws_0, cx_0, cy_0)

      box_center_list.append(box_centers_0)

    centers_out = torch.stack(box_center_list, dim = 0)

    out_feats = self.BEV_sample(centers_out, shared_map)

    out = self.refine_boxes(out_feats, centers_in)

    return out

In [76]:
class CenterPoint(nn.Module):

  def __init__(self, w, h, C, num_classes, K=10, hidden_channels = [64, 32], threshold = 0.45, stride = 2):

    super().__init__()

    self.w = w
    self.h = h
    self.C =C
    self.num_classes = num_classes
    self.K = K
    self.threshold = threshold
    self.stride = stride

    self.hidden_channels1 = hidden_channels[0]
    self.hidden_channels2 = hidden_channels[1]

    self.stage1 = CenterPoint_FirstStage(self.w, self.h, self.C, self.num_classes, self.K, hidden_channels = self.hidden_channels1, stride = self.stride)
    self.stage2 = CenterPoint_SecondStage(self.w, self.h, self.hidden_channels1, self.num_classes, hidden_channels = self.hidden_channels2)

  def IoU(self, center_a0, center_b0):

    center_a = center_a0.detach().cpu()
    center_b = center_b0.detach().cpu()

    l_a, w_a, h_a = center_a[0], center_a[1], center_a[2]
    x_a, y_a, z_a = center_a[5], center_a[6], center_a[7]
    sin_a, cos_a = center_a[3], center_a[4]

    l_b, w_b, h_b = center_b[0], center_b[1], center_b[2]
    x_b, y_b, z_b = center_b[5], center_b[6], center_b[7]
    sin_b, cos_b = center_b[3], center_b[4]

    scale_a = np.array([l_a, w_a, h_a])
    scale_b = np.array([l_b, w_b, h_b])

    translation_a = np.array([x_a, y_a, z_a])
    translation_b = np.array([x_b, y_b, z_b])

    rotation_a = np.array([[cos_a, -1*sin_a,0],[sin_a, cos_a,0],[0,0,1]])
    rotation_b = np.array([[cos_b, -1*sin_b,0],[sin_b, cos_b,0],[0,0,1]])

    box_a = Box.from_transformation(rotation_a, translation_a, scale_a)
    box_b = Box.from_transformation(rotation_b, translation_b, scale_b)

    iou = IoU3d(box_a, box_b)

    return iou.iou()

  def NMS(self, centers, threshold):

    confidence_scores = centers[..., -1].tolist()
    keep = []
    S = []

    for i in range(centers.shape[0]):
      S.append(centers[i])

    while len(S) > 0:
      S_max = max(confidence_scores)
      j = confidence_scores.index(S_max)
      center_max = S[j].squeeze()
      confidence_scores.pop(j)

      keep.append(center_max)
      S.pop(j)

      remove_indices = []

      for i in range(len(S)):

        iou = self.IoU(center_max, S[i])

        if iou > threshold:
          remove_indices.append(i)

      for index in sorted(remove_indices, reverse=True):
        S.pop(index)
        confidence_scores.pop(index)

    return keep

  def forward(self, feat_map):

    centers = self.stage1(feat_map)

    shared_map = self.stage1.get_shared_feats(feat_map)

    updated_centers = self.stage2(centers, shared_map)

    out = self.NMS(updated_centers, self.threshold)

    return out

check how exactly size parameters are outputted, if not log, then change stage 2 get_box_centers

In [7]:
rng = np.random.default_rng()

test_mat = rng.random((64,50,50))

test_mat = torch.from_numpy(test_mat).to(torch.float)

In [8]:
device = "cuda"

test_mat = test_mat.to(device)

In [77]:
model_final = CenterPoint(50,50,64,4)
model_final = model_final.to(device)

In [78]:
out = model_final(test_mat)

In [79]:
out

[tensor([ 0.2175,  0.0935,  0.0740, -0.2526,  0.9676, 47.8222, 45.8566,  0.1378,
          3.0000,  0.3229], device='cuda:0', grad_fn=<SqueezeBackward0>),
 tensor([ 9.9449e-02,  3.4784e-01,  2.2838e-02, -9.8645e-02,  9.9512e-01,
          4.8008e+01,  3.7686e+01,  6.7526e-02,  2.0000e+00,  3.1630e-01],
        device='cuda:0', grad_fn=<SqueezeBackward0>),
 tensor([ 0.2160,  0.1643,  0.0533, -0.2207,  0.9753, 47.8502, 45.8757,  0.1223,
          0.0000,  0.2831], device='cuda:0', grad_fn=<SqueezeBackward0>),
 tensor([ 2.1787e-01,  1.3811e-01,  4.1187e-02, -1.8445e-01,  9.8284e-01,
          4.7830e+01,  4.5824e+01,  3.8433e-02,  1.0000e+00,  2.6002e-01],
        device='cuda:0', grad_fn=<SqueezeBackward0>)]